## Load evaluation results

Load the prepared spatiotemporal analysis, extracted object records, method summary, timestamps, and corepoint coordinates required for the evaluation.

In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py4dgeo

repo_dir = Path.cwd().parent if Path.cwd().name == "jupyter" else Path.cwd()

data_dir = repo_dir / "kijkduin"
evaluation_dir = repo_dir / "results" / "kalman_evaluation"

analysis_file = data_dir / "kijkduin.zip"
objects_file = evaluation_dir / "extracted_objects.pkl"
summary_file = evaluation_dir / "method_summary.csv"

for file_path in (analysis_file, objects_file, summary_file):
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {file_path}\n"
            "Run the previous notebooks first."
        )

analysis = py4dgeo.SpatiotemporalAnalysis(str(analysis_file))

with open(objects_file, "rb") as file:
    extracted_objects = pickle.load(file)

method_summary = pd.read_csv(summary_file)

timestamps_analysis = [
    analysis.reference_epoch.timestamp + time_delta
    for time_delta in analysis.timedeltas
]

corepoints = np.asarray(analysis.corepoints.cloud)

display(method_summary)

## Object correspondence

Define helper functions to measure temporal and spatial overlap between extracted objects. Two objects correspond when they share at least one corepoint and one activity epoch.

In [ ]:
def temporal_overlap_epochs(object_a, object_b):
    """Return the number of overlapping activity epochs."""
    return int(
        max(
            0,
            min(object_a["end_epoch"], object_b["end_epoch"])
            - max(object_a["start_epoch"], object_b["start_epoch"])
            + 1,
        )
    )


def spatial_overlap_count(object_a, object_b):
    """Return the number of shared corepoints."""
    return len(set(object_a["indices"]) & set(object_b["indices"]))


def objects_overlap(object_a, object_b):
    """Return whether two objects overlap spatially and temporally."""
    return (
        spatial_overlap_count(object_a, object_b) > 0
        and temporal_overlap_epochs(object_a, object_b) > 0
    )

## Cross-method object correspondences

Compare each 4D-OBC object with objects from the four Kalman variants. Candidate correspondences must overlap in both space and time and are ranked descriptively by temporal overlap, spatial overlap, event magnitude, and seed duration.

In [ ]:
comparison_methods = [
    "KF-Mag",
    "KF-Rate",
    "KF-Mag-NMS",
    "KF-Rate-NMS",
]

objects_4dobc = extracted_objects["4DOBC"]

seed_tables = {
    "KF-Mag": pd.read_csv(evaluation_dir / "kf_mag_seeds.csv"),
    "KF-Rate": pd.read_csv(evaluation_dir / "kf_rate_seeds.csv"),
    "KF-Mag-NMS": pd.read_csv(evaluation_dir / "kf_mag_nms_seeds.csv"),
    "KF-Rate-NMS": pd.read_csv(evaluation_dir / "kf_rate_nms_seeds.csv"),
}

all_match_records = []

for method_name in comparison_methods:
    candidate_objects = extracted_objects[method_name]
    seed_table = seed_tables[method_name]

    for object_4dobc in objects_4dobc:
        candidate_records = []

        for candidate_object in candidate_objects:
            temporal_overlap = temporal_overlap_epochs(
                object_4dobc, candidate_object
            )
            spatial_overlap = spatial_overlap_count(
                object_4dobc, candidate_object
            )

            if temporal_overlap == 0 or spatial_overlap == 0:
                continue

            mask = (
                (
                    seed_table["corepoint_index_python"]
                    == int(candidate_object["seed_corepoint"])
                )
                & (
                    seed_table["start_epoch"]
                    == int(candidate_object["seed_start_epoch"])
                )
                & (
                    seed_table["end_epoch"]
                    == int(candidate_object["seed_end_epoch"])
                )
            )

            matches = seed_table[mask]
            event_magnitude = (
                float(matches.iloc[0]["event_magnitude"])
                if not matches.empty
                else np.nan
            )

            candidate_records.append(
                {
                    "object_4dobc": int(object_4dobc["object_number"]),
                    "candidate_method": method_name,
                    "candidate_object": int(candidate_object["object_number"]),
                    "temporal_overlap_epochs": temporal_overlap,
                    "spatial_overlap_corepoints": spatial_overlap,
                    "event_magnitude": event_magnitude,
                    "seed_duration_epochs": int(
                        candidate_object["seed_duration_epochs"]
                    ),
                }
            )

        if not candidate_records:
            continue

        candidate_table = pd.DataFrame(candidate_records).sort_values(
            [
                "temporal_overlap_epochs",
                "spatial_overlap_corepoints",
                "event_magnitude",
                "seed_duration_epochs",
            ],
            ascending=False,
        )

        candidate_table["correspondence_rank"] = np.arange(
            1, len(candidate_table) + 1
        )

        all_match_records.extend(
            candidate_table.to_dict(orient="records")
        )

correspondence_table = pd.DataFrame(all_match_records)

display(correspondence_table.head(20))

## Top object correspondences

Select the highest-ranked correspondence for each 4D-OBC object and Kalman method, then save the resulting table for later evaluation.

In [ ]:
top_correspondences = (
    correspondence_table[
        correspondence_table["correspondence_rank"] == 1
    ]
    .sort_values(["object_4dobc", "candidate_method"])
    .reset_index(drop=True)
)

display(top_correspondences)

top_correspondences.to_csv(
    evaluation_dir / "top_object_correspondences.csv",
    index=False,
)

## Seed and object counts

Compare the number of detected seeds and extracted objects produced by the baseline and Kalman-based methods.

In [ ]:
display(method_summary)

plt.figure(figsize=(8, 4))
plt.bar(method_summary["method"], method_summary["n_seeds"])
plt.xlabel("Method")
plt.ylabel("Number of detected seeds")
plt.title("Detected seed count by method")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(method_summary["method"], method_summary["n_objects"])
plt.xlabel("Method")
plt.ylabel("Number of extracted objects")
plt.title("Extracted object count by method")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Object correspondence summary

Summarize how many 4D-OBC objects have a corresponding object for each Kalman method, together with their mean temporal and spatial overlap. These measures describe agreement between methods rather than validation accuracy.

In [ ]:
n_4dobc_objects = len(extracted_objects["4DOBC"])

correspondence_summary = (
    top_correspondences
    .groupby("candidate_method")
    .agg(
        n_corresponding_objects=("object_4dobc", "nunique"),
        mean_temporal_overlap_epochs=("temporal_overlap_epochs", "mean"),
        mean_spatial_overlap_corepoints=("spatial_overlap_corepoints", "mean"),
    )
    .reset_index()
)

correspondence_summary["n_4dobc_objects"] = n_4dobc_objects
correspondence_summary["correspondence_fraction"] = (
    correspondence_summary["n_corresponding_objects"] / n_4dobc_objects
)

display(correspondence_summary)

correspondence_summary.to_csv(
    evaluation_dir / "object_correspondence_summary.csv",
    index=False,
)

## Object correspondence structure

Examine whether multiple 4D-OBC objects correspond to the same Kalman object. This provides a descriptive indication of possible object merging or broader segmentation by the Kalman-based methods.

In [ ]:
mapping_table = top_correspondences[
    ["object_4dobc", "candidate_method", "candidate_object"]
].copy()

merging_table = (
    mapping_table
    .groupby(["candidate_method", "candidate_object"])
    .agg(n_4dobc_objects=("object_4dobc", "nunique"))
    .reset_index()
)

merging_summary = (
    merging_table
    .groupby("candidate_method")
    .agg(
        n_corresponding_kalman_objects=("candidate_object", "nunique"),
        mean_4dobc_objects_per_kalman_object=("n_4dobc_objects", "mean"),
        max_4dobc_objects_per_kalman_object=("n_4dobc_objects", "max"),
        n_kalman_objects_matching_multiple_4dobc=(
            "n_4dobc_objects",
            lambda values: int((values > 1).sum()),
        ),
    )
    .reset_index()
)

display(merging_summary)

merging_summary.to_csv(
    evaluation_dir / "object_merging_summary.csv",
    index=False,
)

## Temporal boundary differences

Compare the start epoch, end epoch, and duration of each top Kalman correspondence with its corresponding 4D-OBC object. Positive boundary differences indicate that the Kalman object starts or ends later. These differences describe temporal segmentation behavior rather than accuracy.

In [ ]:
object_lookup = {
    method_name: {
        int(obj["object_number"]): obj
        for obj in objects
    }
    for method_name, objects in extracted_objects.items()
}

temporal_comparison = top_correspondences.copy()

boundary_records = []

for _, row in temporal_comparison.iterrows():
    object_4dobc = object_lookup["4DOBC"][int(row["object_4dobc"])]
    candidate_object = object_lookup[row["candidate_method"]][
        int(row["candidate_object"])
    ]

    boundary_records.append(
        {
            "object_4dobc_start_epoch": int(object_4dobc["start_epoch"]),
            "object_4dobc_end_epoch": int(object_4dobc["end_epoch"]),
            "candidate_start_epoch": int(candidate_object["start_epoch"]),
            "candidate_end_epoch": int(candidate_object["end_epoch"]),
        }
    )

boundary_table = pd.DataFrame(boundary_records)
temporal_comparison = pd.concat(
    [temporal_comparison.reset_index(drop=True), boundary_table],
    axis=1,
)

temporal_comparison["start_epoch_difference"] = (
    temporal_comparison["candidate_start_epoch"]
    - temporal_comparison["object_4dobc_start_epoch"]
)

temporal_comparison["end_epoch_difference"] = (
    temporal_comparison["candidate_end_epoch"]
    - temporal_comparison["object_4dobc_end_epoch"]
)

temporal_comparison["object_4dobc_duration_epochs"] = (
    temporal_comparison["object_4dobc_end_epoch"]
    - temporal_comparison["object_4dobc_start_epoch"]
    + 1
)

temporal_comparison["candidate_duration_epochs"] = (
    temporal_comparison["candidate_end_epoch"]
    - temporal_comparison["candidate_start_epoch"]
    + 1
)

temporal_comparison["duration_difference_epochs"] = (
    temporal_comparison["candidate_duration_epochs"]
    - temporal_comparison["object_4dobc_duration_epochs"]
)

temporal_comparison["abs_start_epoch_difference"] = (
    temporal_comparison["start_epoch_difference"].abs()
)

temporal_comparison["abs_end_epoch_difference"] = (
    temporal_comparison["end_epoch_difference"].abs()
)

temporal_boundary_summary = (
    temporal_comparison
    .groupby("candidate_method")
    .agg(
        mean_start_difference=("start_epoch_difference", "mean"),
        median_start_difference=("start_epoch_difference", "median"),
        mean_end_difference=("end_epoch_difference", "mean"),
        median_end_difference=("end_epoch_difference", "median"),
        mean_duration_difference=("duration_difference_epochs", "mean"),
        median_duration_difference=("duration_difference_epochs", "median"),
        mean_abs_start_difference=("abs_start_epoch_difference", "mean"),
        mean_abs_end_difference=("abs_end_epoch_difference", "mean"),
    )
    .reset_index()
)

display(temporal_boundary_summary)

temporal_comparison.to_csv(
    evaluation_dir / "temporal_boundary_comparison.csv",
    index=False,
)

temporal_boundary_summary.to_csv(
    evaluation_dir / "temporal_boundary_summary.csv",
    index=False,
)

## Object fragmentation structure

Count how many Kalman objects overlap each 4D-OBC object in both space and time. Multiple overlapping candidate objects indicate a more fragmented correspondence structure for that method.

In [ ]:
fragmentation_table = (
    correspondence_table
    .groupby(["candidate_method", "object_4dobc"])
    .agg(
        n_overlapping_candidate_objects=("candidate_object", "nunique"),
        overlapping_candidate_objects=(
            "candidate_object",
            lambda values: ", ".join(map(str, sorted(set(values)))),
        ),
        maximum_temporal_overlap_epochs=("temporal_overlap_epochs", "max"),
        maximum_spatial_overlap_corepoints=("spatial_overlap_corepoints", "max"),
    )
    .reset_index()
)

fragmentation_summary = (
    fragmentation_table
    .groupby("candidate_method")
    .agg(
        n_4dobc_objects_with_correspondence=("object_4dobc", "nunique"),
        mean_candidates_per_4dobc_object=(
            "n_overlapping_candidate_objects",
            "mean",
        ),
        max_candidates_per_4dobc_object=(
            "n_overlapping_candidate_objects",
            "max",
        ),
        n_4dobc_objects_with_multiple_candidates=(
            "n_overlapping_candidate_objects",
            lambda values: int((values > 1).sum()),
        ),
    )
    .reset_index()
)

display(fragmentation_summary)

fragmentation_table.to_csv(
    evaluation_dir / "object_fragmentation_table.csv",
    index=False,
)

fragmentation_summary.to_csv(
    evaluation_dir / "object_fragmentation_summary.csv",
    index=False,
)

## Visual comparison of corresponding objects

Visualize one 4D-OBC object and its top correspondence from each Kalman method. The temporal curves show the same temporally smoothed M3C2 signal averaged over the spatial support of each object, while the shaded intervals show their detected activity periods. Convex hulls provide a simplified visualization of spatial extent.

In [ ]:
import matplotlib.dates as mdates
from scipy.spatial import ConvexHull

object_number = 1

if object_number < 1:
    raise ValueError("object_number must start from 1.")

methods_to_compare = [
    "KF-Mag",
    "KF-Rate",
    "KF-Mag-NMS",
    "KF-Rate-NMS",
]


def draw_object_outline(ax, indices, linestyle, label):
    """Draw a convex hull around an object's corepoints."""
    xy = corepoints[np.asarray(indices, dtype=int), :2]
    xy = xy[np.all(np.isfinite(xy), axis=1)]
    xy = np.unique(xy, axis=0)

    if len(xy) < 3:
        ax.scatter(xy[:, 0], xy[:, 1], s=8, label=label)
        return

    hull = ConvexHull(xy)
    hull_xy = xy[hull.vertices]
    hull_xy = np.vstack([hull_xy, hull_xy[0]])

    ax.plot(
        hull_xy[:, 0],
        hull_xy[:, 1],
        linestyle=linestyle,
        linewidth=2.2,
        label=label,
    )


object_4dobc = object_lookup["4DOBC"][object_number]
indices_4dobc = np.asarray(object_4dobc["indices"], dtype=int)

selected_correspondences = top_correspondences[
    top_correspondences["object_4dobc"] == object_number
].copy()

if selected_correspondences.empty:
    raise ValueError(
        f"No spatiotemporal correspondence found for "
        f"4DOBC object {object_number}."
    )

signal_data = np.asarray(analysis.smoothed_distances)

mean_ts_4dobc = np.nanmean(
    signal_data[indices_4dobc],
    axis=0,
)

start_4dobc = int(object_4dobc["start_epoch"])
end_4dobc = int(object_4dobc["end_epoch"])

fig, axes = plt.subplots(
    len(methods_to_compare),
    2,
    figsize=(15, 4.5 * len(methods_to_compare)),
)

for row_index, method_name in enumerate(methods_to_compare):
    ax_time, ax_space = axes[row_index]

    method_rows = selected_correspondences[
        selected_correspondences["candidate_method"] == method_name
    ]

    if method_rows.empty:
        for ax in (ax_time, ax_space):
            ax.text(
                0.5,
                0.5,
                "No spatiotemporal correspondence",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
        continue

    correspondence = method_rows.iloc[0]
    candidate_number = int(correspondence["candidate_object"])
    candidate_object = object_lookup[method_name][candidate_number]

    candidate_indices = np.asarray(candidate_object["indices"], dtype=int)
    candidate_start = int(candidate_object["start_epoch"])
    candidate_end = int(candidate_object["end_epoch"])

    candidate_mean_ts = np.nanmean(
        signal_data[candidate_indices],
        axis=0,
    )

    ax_time.plot(
        timestamps_analysis,
        mean_ts_4dobc,
        linewidth=2,
        label=f"4DOBC {object_number}",
    )
    ax_time.plot(
        timestamps_analysis,
        candidate_mean_ts,
        linewidth=2,
        linestyle="--",
        label=f"{method_name} {candidate_number}",
    )

    ax_time.axvspan(
        timestamps_analysis[start_4dobc],
        timestamps_analysis[end_4dobc],
        alpha=0.12,
        label="4DOBC interval",
    )
    ax_time.axvspan(
        timestamps_analysis[candidate_start],
        timestamps_analysis[candidate_end],
        alpha=0.12,
        label=f"{method_name} interval",
    )

    ax_time.xaxis.set_major_formatter(mdates.DateFormatter("%b-%d"))
    ax_time.tick_params(axis="x", rotation=15)
    ax_time.set_xlabel("Date")
    ax_time.set_ylabel("Smoothed M3C2 distance [m]")
    ax_time.set_title(
        f"4DOBC {object_number} vs {method_name} {candidate_number}\n"
        f"Temporal overlap: "
        f"{int(correspondence['temporal_overlap_epochs'])} epochs"
    )
    ax_time.legend(fontsize=8)
    ax_time.grid(alpha=0.2)

    intersection_indices = np.intersect1d(
        indices_4dobc,
        candidate_indices,
    )

    if len(intersection_indices) > 0:
        intersection_xy = corepoints[intersection_indices, :2]
        ax_space.scatter(
            intersection_xy[:, 0],
            intersection_xy[:, 1],
            s=5,
            alpha=0.6,
            label=f"Shared corepoints (n={len(intersection_indices)})",
        )

    draw_object_outline(
        ax_space,
        indices_4dobc,
        "-",
        f"4DOBC {object_number}",
    )
    draw_object_outline(
        ax_space,
        candidate_indices,
        "--",
        f"{method_name} {candidate_number}",
    )

    ax_space.set_aspect("equal", adjustable="box")
    ax_space.set_xlabel("X [m]")
    ax_space.set_ylabel("Y [m]")
    ax_space.set_title(
        "Spatial comparison — "
        f"shared corepoints: "
        f"{int(correspondence['spatial_overlap_corepoints'])}"
    )
    ax_space.legend(fontsize=8)
    ax_space.grid(alpha=0.15)

fig.suptitle(
    f"Object correspondence — 4DOBC object {object_number}",
    fontsize=15,
)

plt.tight_layout()
plt.show()

display_columns = [
    "candidate_method",
    "candidate_object",
    "temporal_overlap_epochs",
    "spatial_overlap_corepoints",
    "event_magnitude",
    "seed_duration_epochs",
]

display(selected_correspondences[display_columns])

## Kalman signal diagnostic

Inspect the signal-level effect of Kalman filtering at the seed corepoint of a representative corresponding object. Raw M3C2, standard temporal smoothing, Kalman-smoothed change, and Kalman-estimated change rate are compared together with their uncertainty. Rate epochs are considered significant when the 95% confidence interval does not contain zero.

In [ ]:
object_number = 1
kalman_method = "KF-Mag"
z_threshold = 1.96

valid_methods = [
    "KF-Mag",
    "KF-Rate",
    "KF-Mag-NMS",
    "KF-Rate-NMS",
]

if kalman_method not in valid_methods:
    raise ValueError(f"kalman_method must be one of: {valid_methods}")

selected_row = top_correspondences[
    (top_correspondences["object_4dobc"] == object_number)
    & (top_correspondences["candidate_method"] == kalman_method)
]

if selected_row.empty:
    raise ValueError(
        f"No correspondence found for 4DOBC object {object_number} "
        f"and method {kalman_method}."
    )

candidate_number = int(selected_row.iloc[0]["candidate_object"])
candidate_object = object_lookup[kalman_method][candidate_number]

corepoint_index = int(candidate_object["seed_corepoint"])
candidate_start = int(candidate_object["start_epoch"])
candidate_end = int(candidate_object["end_epoch"])

raw_m3c2 = np.asarray(analysis.distances)[corepoint_index].copy()

if analysis.smoothed_distances is None:
    raise ValueError("Temporal smoothing from Notebook 1 is not available.")

standard_smoothed_distances = np.asarray(
    analysis.smoothed_distances
).copy()

temporal_smoothed_m3c2 = standard_smoothed_distances[
    corepoint_index
].copy()

kalman_diagnostic = py4dgeo.KalmanRegionGrowingAlgorithm(
    detection_mode="magnitude",
    process_sigma=0.01,
    min_sigma_obs=0.005,
    kalman_cache_path=repo_dir / "results" / "kalman_cache",
)

kalman_diagnostic._analysis = analysis

try:
    kalman_diagnostic._run_kalman_filter()

    kalman_change = np.asarray(
        kalman_diagnostic.kalman_change[corepoint_index]
    ).copy()

    kalman_rate = np.asarray(
        kalman_diagnostic.kalman_rate[corepoint_index]
    ).copy()

    sigma_change = np.asarray(
        kalman_diagnostic.kalman_sigma_change[corepoint_index]
    ).copy()

    sigma_rate = np.asarray(
        kalman_diagnostic.kalman_sigma_rate[corepoint_index]
    ).copy()

finally:
    analysis.smoothed_distances = standard_smoothed_distances

change_lower = kalman_change - z_threshold * sigma_change
change_upper = kalman_change + z_threshold * sigma_change

rate_lower = kalman_rate - z_threshold * sigma_rate
rate_upper = kalman_rate + z_threshold * sigma_rate

significant_positive_rate = rate_lower > 0
significant_negative_rate = rate_upper < 0
significant_rate = significant_positive_rate | significant_negative_rate

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
ax_change, ax_rate = axes

ax_change.plot(
    timestamps_analysis,
    raw_m3c2,
    linewidth=1,
    alpha=0.55,
    label="Raw M3C2",
)
ax_change.plot(
    timestamps_analysis,
    temporal_smoothed_m3c2,
    linewidth=2,
    label="Temporal smoothing",
)
ax_change.plot(
    timestamps_analysis,
    kalman_change,
    linewidth=2.2,
    label="Kalman change",
)
ax_change.fill_between(
    timestamps_analysis,
    change_lower,
    change_upper,
    alpha=0.15,
    label=f"Kalman change ± {z_threshold:.2f}σ",
)
ax_change.axvspan(
    timestamps_analysis[candidate_start],
    timestamps_analysis[candidate_end],
    alpha=0.10,
    label=f"{kalman_method} object interval",
)
ax_change.axhline(0, linewidth=1, linestyle="--")
ax_change.set_ylabel("Surface change [m]")
ax_change.set_title(f"Change signal at corepoint {corepoint_index}")
ax_change.grid(alpha=0.2)
ax_change.legend(fontsize=9)

ax_rate.plot(
    timestamps_analysis,
    kalman_rate,
    linewidth=2,
    label="Kalman rate",
)
ax_rate.fill_between(
    timestamps_analysis,
    rate_lower,
    rate_upper,
    alpha=0.18,
    label=f"Kalman rate ± {z_threshold:.2f}σ",
)
ax_rate.axhline(0, linewidth=1, linestyle="--")

if np.any(significant_rate):
    ax_rate.scatter(
        np.asarray(timestamps_analysis)[significant_rate],
        kalman_rate[significant_rate],
        s=18,
        label="Significant rate",
        zorder=5,
    )

ax_rate.axvspan(
    timestamps_analysis[candidate_start],
    timestamps_analysis[candidate_end],
    alpha=0.10,
    label=f"{kalman_method} object interval",
)
ax_rate.set_xlabel("Date")
ax_rate.set_ylabel("Change rate [m/day]")
ax_rate.set_title("Kalman-estimated change rate")
ax_rate.xaxis.set_major_formatter(mdates.DateFormatter("%b-%d"))
ax_rate.tick_params(axis="x", rotation=15)
ax_rate.grid(alpha=0.2)
ax_rate.legend(fontsize=9)

fig.suptitle(
    f"Kalman signal diagnostic — {kalman_method} object "
    f"{candidate_number}, 4DOBC correspondence {object_number}",
    fontsize=14,
)

plt.tight_layout()
plt.show()

diagnostic_summary = pd.DataFrame(
    {
        "metric": [
            "Representative corepoint",
            "Kalman object",
            "Object duration [epochs]",
            "Significant positive-rate epochs",
            "Significant negative-rate epochs",
            "Total significant-rate epochs",
        ],
        "value": [
            corepoint_index,
            candidate_number,
            candidate_end - candidate_start + 1,
            int(significant_positive_rate.sum()),
            int(significant_negative_rate.sum()),
            int(significant_rate.sum()),
        ],
    }
)

display(diagnostic_summary)